# Overview

This notebook is to understand the logic behind the `publishing.py` script stored in databricks before wrapping it in this package. The idea is to get avoid duplicating stuff we may have developed somewhere else.

# Setup

In [8]:
import json
import re
import requests
import os

In [9]:
# Import from token file
with open('../../me_keys.token', 'r') as file:
	api_key = file.read().strip().split('=')[1]

In [10]:
# Define globals
meta_url = 'https://metadataeditor.worldbank.org/index.php'

# Understanding

## normalize_indicator_input

This functions is defined within multiple notebooks and can be stored in a utils module in this package.

In [2]:
def normalize_indicator_input(raw_value):
    if not raw_value:
        return []

    raw_value = raw_value.strip()

    # Case 1: JSON array (from for_each_task)
    if raw_value.startswith("[") and raw_value.endswith("]"):
        try:
            values = json.loads(raw_value)
            return [str(v).strip().strip('"').strip("'") for v in values]
        except Exception:
            pass  # fall through

    # Case 2: Comma-separated string (job-level)
    parts = raw_value.split(",")

    return [
        re.sub(r'^[\'"]+|[\'"]+$', "", p.strip())
        for p in parts
        if p.strip()
    ]

In [6]:
print(normalize_indicator_input('WB_ASDF'))
print(normalize_indicator_input('WB_ASDF,WB_QWER'))
print(normalize_indicator_input('[WB_ASDF]'))
print(normalize_indicator_input('[WB_ASDF,WB_QWER]'))

['WB_ASDF']
['WB_ASDF', 'WB_QWER']
['[WB_ASDF]']
['[WB_ASDF', 'WB_QWER]']


## get_meta_by_dataset_db

### function

In [ ]:
def get_meta_by_dataset_db(meta_url, dataset_id, api_key, type, collection_num):
    """
    Extract Data360 metadata projects belonging to a dataset.

    :param meta_url: Base URL of metadata API
    :param dataset_id: Dataset identifier
    :param api_key: API key
    :param type: "timeseries" or "timeseries-db"
    :return: DataFrame of metadata projects
    """
    limit = 1000  # Max limit per page
    headers = {"x-api-key": api_key}

    request_url = f"{meta_url}/api/editor?collection={collection_num}&type={type}&limit={limit}&offset=0"
    response = requests.get(request_url, headers=headers)
    result = response.json()

    if response.status_code != 200:
        raise RuntimeError(f"Metadata request failed: {response.text}")

    total_cases = result.get("total", 0)
    exist_id = {}

    for i in range(0, total_cases, limit):
        paginated_url = f"{meta_url}/api/editor?collection={collection_num}&type={type}&limit={limit}&offset={i}"
        paginated_response = requests.get(paginated_url, headers=headers)
        paginated_result = paginated_response.json()
        projects = paginated_result.get("projects", [])

        if type == "timeseries":
            for item in projects:
                if not isinstance(item, dict):
                    continue

                attrs = item.get("attributes")
                if not isinstance(attrs, dict):
                    continue    

                if attrs.get("database_id") == dataset_id.upper():
                    exist_id[item["study_idno"]] = item["id"]
                    print(f">>>>> Found existing Data360 metadata project for indicator {item["id"]}!")

        elif type == "timeseries-db":
            for item in projects:
                if not isinstance(item, dict):
                    continue

                if item.get("study_idno") == dataset_id:
                    exist_id[item["study_idno"]] = item["id"]
                    print(f">>>>> Found existing Data360 metadata project for dataset {dataset_id}!")

    if not exist_id:
        raise Exception(">>>>> No existing Data360 metadata projects found! Check your Metadata Editor account role.")
    else:
        print(">>>>> Existing Data360 metadata projects found!")
        meta_id = (
            pd.DataFrame.from_dict(exist_id, orient="index", columns=["idno"])
            .reset_index()
        )
        return meta_id

### line-by-line

In [11]:
limit = 1000  # Max limit per page
headers = {"x-api-key": api_key}

In [ ]:
request_url = f"{meta_url}/api/editor?collection={collection_num}&type={type}&limit={limit}&offset=0"
    response = requests.get(request_url, headers=headers)
    result = response.json()

## export_metadata_ind

In [ ]:
def export_metadata_ind(
    meta_url: str,
    dataset_id: str,
    indicator_list: list,
    api_key: str,
    catalog_env: str,
    obs_conf_path: str,
    collection_num: int
):
    # Define root folder
    root_folder = f"/Volumes/{catalog_env}_data360/volumes/data360-dropzone/{obs_conf_path}/datasets/{dataset_id}/metadata"
    folders = {
        "json": root_folder,
        "pdf": f"{root_folder}/download",
        "dataset_json": f"{root_folder}/dataset",
        "dataset_pdf": f"{root_folder}/dataset/download",
    }

	# Create directories, ignore if they already exist
    for f in folders.values():
        os.makedirs(f, exist_ok=True)

    # Get metadata
    print(">>> Fetching indicator metadata...")
    df_indicators = get_meta_by_dataset_db(meta_url, dataset_id, api_key, "timeseries", collection_num)
    print(">>> Fetching database metadata...")
    df_databases = get_meta_by_dataset_db(meta_url, dataset_id, api_key, "timeseries-db", collection_num)

    if not indicator_list:
        pass
    else:
        df_indicators = df_indicators[df_indicators['index'].isin(indicator_list)]

    headers = {"x-api-key": api_key}
    export_results = {"json": [], "pdf": []}
    json_exported, pdf_exported = {}, {}

    # Loop through databases + indicators
    for df, j_folder, p_folder in [
        (df_databases, folders["dataset_json"], folders["dataset_pdf"]),
        (df_indicators, folders["json"], folders["pdf"])
    ]:
        for _, row in df.iterrows():
            obj_id = row["idno"]
            obj_name = row["index"]   
            print(f">>>>>>>>> Processing {obj_name}: {obj_id}...")

            result = export_file(meta_url, obj_id, obj_name, headers, j_folder, p_folder)

            if isinstance(result["json"], str) and result["json"].endswith(".json"):
                export_results["json"].append(result["json"])
                json_exported[obj_name] = True
            else:
                json_exported[obj_name] = result["json"]

            if isinstance(result["pdf"], str) and result["pdf"].endswith(".pdf"):
                export_results["pdf"].append(result["pdf"])
                pdf_exported[obj_name] = True
            else:
                pdf_exported[obj_name] = result["pdf"]

            time.sleep(0.1)  # light rate limiting

    print(f"✅ Metadata export for dataset {dataset_id} completed.")
    return export_results, json_exported, pdf_exported